In [1]:
import cv2
import sys
from pathlib import Path
import numpy as np
import open3d as o3d
import yaml
import json
from ast import literal_eval
from PIL import Image
import torch
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts').is_dir():
    raise RuntimeError('Start Jupyter from the repository root')
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from multi_view_scan.aux_math import *
from scene_reconstruction.o3d_process import rgbd_to_pcd, rgbd_to_pcd_mask, remove_outlier_o3d
from scene_reconstruction.vlm_api import OBJ_LIST, make_promt_return_all_object, make_prompt_from_user_input
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, SamModel, SamProcessor

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Load VLM Models

In [2]:
QWEN_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
QWEN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QWEN_DTYPE = torch.bfloat16 if QWEN_DEVICE == "cuda" else torch.float32
SAM_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID, dtype=QWEN_DTYPE, device_map=QWEN_DEVICE
)
qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
print(f"Qwen3-VL loaded on: {QWEN_DEVICE}")

SAM_MODEL_ID = "facebook/sam-vit-base"
sam_processor = SamProcessor.from_pretrained(SAM_MODEL_ID)
sam_model = SamModel.from_pretrained(SAM_MODEL_ID).to(SAM_DEVICE)
print(f"SAM loaded on: {SAM_DEVICE}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Qwen3-VL loaded on: cuda


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

SAM loaded on: cuda


In [3]:

def run_qwen_vl(image, prompt, max_new_tokens=512):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    inputs = qwen_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to(qwen_model.device)
    with torch.no_grad():
        generated_ids = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    return qwen_processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

def _parse_qwen_structure(text):
    cleaned = text.strip()
    fence = chr(96) * 3
    if cleaned.startswith(fence):
        cleaned = cleaned.split("\n", 1)[-1]
    if cleaned.endswith(fence):
        cleaned = cleaned[:-3].strip()
    starts = [i for i in (cleaned.find("["), cleaned.find("{")) if i >= 0]
    if starts:
        start = min(starts)
        end = max(cleaned.rfind("]"), cleaned.rfind("}"))
        if end >= start:
            cleaned = cleaned[start:end + 1]
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return literal_eval(cleaned)

def find_objects_with_prompt(image_path, prompt):
    image = Image.open(image_path).convert("RGB")
    result = _parse_qwen_structure(run_qwen_vl(image, prompt, max_new_tokens=256))
    if isinstance(result, dict):
        result = result.get("objects", [])
    if not isinstance(result, list) or not all(isinstance(name, str) for name in result):
        raise ValueError(f"Qwen3-VL returned an invalid object list: {result!r}")
    valid_names = {name.replace("_", " ").lower(): name for name in OBJ_LIST}
    normalized = [name.replace("_", " ").lower() for name in result]
    return [valid_names[name] for name in normalized if name in valid_names]

def find_bounding_boxes_qwen(image_path, object_names):
    if not object_names:
        return []
    image = Image.open(image_path).convert("RGB")
    prompt = (
        f"Find these objects in the image: {json.dumps(object_names)}. "
        "Return only a JSON list with one item per visible object: "
        "[{\"bbox_2d\": [xmin, ymin, xmax, ymax], \"label\": \"exact object name\"}]. "
        "Use normalized integer coordinates from 0 to 1000 with the origin at "
        "the top-left. Omit objects that are not visible. Do not add explanations."
    )
    result = _parse_qwen_structure(run_qwen_vl(image, prompt))
    if isinstance(result, dict):
        result = result.get("objects", result.get("detections", [result]))
    if not isinstance(result, list):
        raise ValueError(f"Qwen3-VL returned invalid detections: {result!r}")

    width, height = image.size
    canonical = {name.replace("_", " ").lower(): name for name in object_names}
    detections = []
    for item in result:
        if not isinstance(item, dict):
            continue
        label = str(item.get("label", item.get("object", "")))
        object_name = canonical.get(label.replace("_", " ").lower())
        box = item.get("bbox_2d", item.get("bbox"))
        if object_name is None or not isinstance(box, (list, tuple)) or len(box) != 4:
            continue
        x1, y1, x2, y2 = (float(value) for value in box)
        x1, x2 = sorted((np.clip(x1, 0, 1000) * width / 1000, np.clip(x2, 0, 1000) * width / 1000))
        y1, y2 = sorted((np.clip(y1, 0, 1000) * height / 1000, np.clip(y2, 0, 1000) * height / 1000))
        if x2 > x1 and y2 > y1:
            detections.append({"object": object_name, "bbox": [x1, y1, x2, y2]})
    return detections

def objects_detection_qwen(image_path, object_names):
    base_image = Image.open(image_path).convert("RGB")
    detections = find_bounding_boxes_qwen(image_path, object_names)
    results = []
    for detection in detections:
        box = detection["bbox"]
        sam_inputs = sam_processor(
            images=base_image, input_boxes=[[box]], return_tensors="pt"
        )
        sam_inputs = {name: value.to(SAM_DEVICE) for name, value in sam_inputs.items()}
        with torch.no_grad():
            sam_outputs = sam_model(**sam_inputs)
        masks = sam_processor.post_process_masks(
            sam_outputs.pred_masks.cpu(), sam_inputs["original_sizes"].cpu(),
            sam_inputs["reshaped_input_sizes"].cpu()
        )[0]
        best_mask = int(sam_outputs.iou_scores[0, 0].argmax().item())
        mask = (masks[0, best_mask].numpy() > 0).astype(np.uint8) * 255
        x1, y1, x2, y2 = box
        results.append({
            "object": detection["object"],
            "center": ((x1 + x2) / 2, (y1 + y2) / 2),
            "bbox": box,
            "mask": mask,
        })
    return results


In [ ]:
OBJ_LIST = [
    "Apple",
    "Banana",
    "Brick",
    "Biscuit_Box",
    "Hammer",
    "Clamp",
    "Cup",
    "Mustard_Bottle",
    "Pear",
    "Power_Drill",
    "Screwdriver",
    "Scissor",
    "Strawberry",
    "Tennis_Ball",
    "Soup_Can",
    "Strawberry",
]

### Test models and functions

In [5]:
# Image and object selection with Qwen3-VL
image_path = "/home/hier-tony/Projects/vla_dual_arm/scan_output/steps/step_003/right_color.png"
user_input = "I want some fruit"

# prompt = make_promt_return_all_object(OBJ_LIST)
prompt = make_prompt_from_user_input(OBJ_LIST, user_input)
object_names = find_objects_with_prompt(image_path, prompt)
print(object_names)

bbox_result = find_bounding_boxes_qwen(image_path, object_names)
print(bbox_result)

detect_result = objects_detection_qwen(image_path, object_names)

[]
[]


In [6]:
# cv2.imshow("Mask", detect_result[0]["mask"])
# cv2.waitKey(0)
# cv2.destroyAllWindows()

## Vision pipeline for segmentation

In [13]:
from collections import defaultdict
from pathlib import Path

SCAN_DIR = Path("/home/hier-tony/Projects/vla_dual_arm/scan_output")
MANIFEST_PATH = SCAN_DIR / "manifest.json"
CAMERA_YAML = Path("/home/hier-tony/Projects/dual_manipulation_isaac_sim/env/urdf/camera.yaml")
MASK_DIR = SCAN_DIR / "mask_segment"
MASK_DIR.mkdir(parents=True, exist_ok=True)

# Load the scan index and shared camera intrinsics.
manifest = json.loads(MANIFEST_PATH.read_text())
with CAMERA_YAML.open("r") as stream:
    camera_config = yaml.safe_load(stream)
intr = camera_config["intrinsics"]
width, height = (int(value) for value in intr["resolution"])
intrinsic = o3d.camera.PinholeCameraIntrinsic(
    width, height, intr["fx"], intr["fy"], intr["cx"], intr["cy"]
)

# Detect every valid object from the configured object list.
prompt = make_promt_return_all_object(OBJ_LIST)

# Build views only from successful manifest captures. Stale image folders and
# motion-failed entries are intentionally ignored.
captured_views = []
for capture in manifest.get("captures", []):
    if capture.get("status") != "captured":
        continue
    step = int(capture["step"])
    for side in ("left", "right"):
        camera_capture = capture.get(side, {})
        pose = camera_capture.get("actual_camera_pose")
        if pose is None:
            pose = camera_capture.get("desired_camera_pose")
            print(f"warning: step {step:03d} {side} has no actual pose; using desired pose")
        if pose is None or "color_image" not in camera_capture or "depth_image" not in camera_capture:
            print(f"warning: step {step:03d} {side} is incomplete; skipping")
            continue

        color_path = SCAN_DIR / camera_capture["color_image"]
        depth_path = SCAN_DIR / camera_capture["depth_image"]
        if not color_path.is_file() or not depth_path.is_file():
            print(f"warning: step {step:03d} {side} images are missing; skipping")
            continue
        TF_world_cam = matrix_from_pose(
            pose["position_xyz"], pose["quaternion_xyzw"]
        )
        captured_views.append((step, side, color_path, depth_path, TF_world_cam))

print(
    f"Loaded {len(captured_views)} captured RGB-D views in "
    f"{manifest.get('world_frame', 'world')} coordinates"
)

obj_pcds = defaultdict(list)
scene_pcds = []
for view_index, (step, side, color_path, depth_path, TF_world_cam) in enumerate(captured_views):
    image_id = f"step_{step:03d}_{side}"
    color_path_str = str(color_path)
    depth_path_str = str(depth_path)

    object_names = find_objects_with_prompt(color_path_str, prompt)
    detect_results = objects_detection_qwen(color_path_str, object_names)
    for detect_result in detect_results:
        select_mask = detect_result["mask"]
        object_name = detect_result["object"]
        print(f"{image_id}: {object_name}")
        obj_pcd = rgbd_to_pcd_mask(
            color_path_str, depth_path_str, select_mask, TF_world_cam, intrinsic
        )
        obj_pcds[object_name].append(obj_pcd)

        color_img = cv2.imread(color_path_str, cv2.IMREAD_COLOR)
        mask_img = cv2.bitwise_and(color_img, color_img, mask=select_mask)
        cv2.imwrite(str(MASK_DIR / f"{image_id}_{object_name}_normal_mask.png"), select_mask)
        cv2.imwrite(str(MASK_DIR / f"{image_id}_{object_name}_color_mask.png"), mask_img)

    scene_pcd = rgbd_to_pcd(
        color_path_str, depth_path_str, TF_world_cam, intrinsic
    )
    scene_pcds.append(scene_pcd)

Loaded 20 captured RGB-D views in world coordinates
step_001_left: Strawberry
step_001_left: Soup_Can
step_001_left: Mustard_Bottle
step_001_right: Mustard_Bottle
step_001_right: Soup_Can
step_001_right: Strawberry
step_001_right: Power_Drill
step_002_left: Mustard_Bottle
step_002_left: Power_Drill
step_002_left: Strawberry
step_002_left: Soup_Can
step_002_right: Power_Drill
step_002_right: Soup_Can
step_002_right: Strawberry
step_003_left: Mustard_Bottle
step_003_left: Power_Drill
step_003_left: Soup_Can
step_003_right: Mustard_Bottle
step_003_right: Power_Drill
step_003_right: Soup_Can
step_004_left: Mustard_Bottle
step_004_left: Power_Drill
step_004_left: Soup_Can
step_004_left: Cup
step_004_right: Mustard_Bottle
step_004_right: Power_Drill
step_004_right: Soup_Can
step_007_left: Strawberry
step_007_left: Soup_Can
step_007_left: Mustard_Bottle
step_007_right: Cup
step_007_right: Mustard_Bottle
step_007_right: Power_Drill
step_007_right: Power_Drill
step_007_right: Strawberry
step_00

## Visualize the point clouds

In [17]:
# Origin axis coordinate
axis = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=0.1,       # length of the axes (adjust to your scale)
    origin=[0, 0, 0]
)

# 3d scene reconstruction
final_scene_pcd = sum(scene_pcds[1:], scene_pcds[0])
voxel_size = 0.005
final_scene_pcd = final_scene_pcd.voxel_down_sample(voxel_size)
show_scene = [axis, final_scene_pcd]
o3d.visualization.draw_geometries(show_scene)

# Visualize object point clouds
all_obj_pcds = [pcd for object_clouds in obj_pcds.values() for pcd in object_clouds]
if all_obj_pcds:
    final_obj_pcd = sum(all_obj_pcds[1:], all_obj_pcds[0])
    final_obj_pcd = final_obj_pcd.voxel_down_sample(0.005)
    final_obj_pcd, ind = remove_outlier_o3d(final_obj_pcd, nb_neighbors=50, std_ratio=10)
    points = np.asarray(final_obj_pcd.points)
    keep_indices = np.flatnonzero(points[:, 2] >= 0.0)
    final_obj_pcd = final_obj_pcd.select_by_index(keep_indices)
    show_objs = [axis, final_obj_pcd]
    o3d.visualization.draw_geometries(show_objs)
else:
    final_obj_pcd = o3d.geometry.PointCloud()
    print("No Qwen3-VL object detections were available to visualize.")

for object_name, object_clouds in sorted(obj_pcds.items()):
    obj_pcd = o3d.geometry.PointCloud()
    for pcd in object_clouds:
        obj_pcd += pcd
    obj_pcd = obj_pcd.voxel_down_sample(0.001)
    obj_pcd, _ = obj_pcd.remove_statistical_outlier(
            nb_neighbors=min(50, len(obj_pcd.points) - 1),
            std_ratio=2.0,
        )
    points = np.asarray(obj_pcd.points)
    keep_indices = np.flatnonzero(points[:, 2] >= 0.0)
    obj_pcd = obj_pcd.select_by_index(keep_indices)
    o3d.visualization.draw_geometries([axis, obj_pcd], window_name=object_name)


## Grasp pipeline

The following four sections persist reconstructed clouds, select one detected object from a user command, generate its grasp, and optionally execute it.

### 1. Save reconstructed point clouds as JSON

In [18]:
from scene_reconstruction.o3d_process import save_point_cloud_json

POINT_CLOUD_DIR = SCAN_DIR / "pointclouds"
POINT_CLOUD_INDEX_PATH = POINT_CLOUD_DIR / "index.json"
POINT_CLOUD_DIR.mkdir(parents=True, exist_ok=True)
world_frame = manifest.get("world_frame", "world")

# Save the filtered scene cloud first.
scene_json_path = save_point_cloud_json(
    POINT_CLOUD_DIR / "scene.json", final_scene_pcd, frame_id=world_frame
)
pointcloud_index = {
    "format": "scan-point-cloud-index-v1",
    "frame_id": world_frame,
    "scene": str(scene_json_path.relative_to(SCAN_DIR)),
    "objects": {},
}

# Merge all views of each detected object before filtering and saving it.
for object_name, object_clouds in sorted(obj_pcds.items()):
    if not object_clouds:
        continue
    object_cloud = o3d.geometry.PointCloud()
    for view_cloud in object_clouds:
        object_cloud += view_cloud
    object_cloud = object_cloud.voxel_down_sample(0.005)
    if len(object_cloud.points) >= 30:
        object_cloud, _ = object_cloud.remove_statistical_outlier(
            nb_neighbors=min(20, len(object_cloud.points) - 1),
            std_ratio=2.0,
        )
    if not object_cloud.has_points():
        print(f"warning: {object_name} is empty after filtering; skipping")
        continue

    safe_name = object_name.lower().replace(" " , "_").replace("/", "_")
    object_path = save_point_cloud_json(
        POINT_CLOUD_DIR / f"{safe_name}.json",
        object_cloud,
        frame_id=world_frame,
    )
    pointcloud_index["objects"][object_name] = {
        "path": str(object_path.relative_to(SCAN_DIR)),
        "point_count": len(object_cloud.points),
    }

# Write one small index so later sections do not need to scan directories.
POINT_CLOUD_INDEX_PATH.write_text(
    json.dumps(pointcloud_index, indent=2) + "\n", encoding="utf-8"
)
print(f"Saved scene and {len(pointcloud_index['objects'])} object clouds")
print(f"Point-cloud index: {POINT_CLOUD_INDEX_PATH}")
pointcloud_index


Saved scene and 5 object clouds
Point-cloud index: /home/hier-tony/Projects/vla_dual_arm/scan_output/pointclouds/index.json


{'format': 'scan-point-cloud-index-v1',
 'frame_id': 'world',
 'scene': 'pointclouds/scene.json',
 'objects': {'Cup': {'path': 'pointclouds/cup.json', 'point_count': 553},
  'Mustard_Bottle': {'path': 'pointclouds/mustard_bottle.json',
   'point_count': 2596},
  'Power_Drill': {'path': 'pointclouds/power_drill.json', 'point_count': 2212},
  'Soup_Can': {'path': 'pointclouds/soup_can.json', 'point_count': 1734},
  'Strawberry': {'path': 'pointclouds/strawberry.json', 'point_count': 307}}}

### 2. Select an available object from the user command

Qwen3-VL can select only an object backed by a saved point cloud.

In [19]:
POINT_CLOUD_INDEX_PATH = SCAN_DIR / "pointclouds" / "index.json"
pointcloud_index = json.loads(POINT_CLOUD_INDEX_PATH.read_text(encoding="utf-8"))
detected_objects = sorted(pointcloud_index.get("objects", {}))
if not detected_objects:
    raise RuntimeError("no saved object point clouds are available for selection")

print(f"Available objects: {detected_objects}")

def select_detected_object(user_command, available_objects):
    """Use the loaded Qwen3-VL model to select one available object."""
    if not user_command.strip():
        raise ValueError("user_command must not be empty")

    # Restrict the model prompt to objects backed by saved point clouds.
    prompt = (
        f"Available objects: {json.dumps(list(available_objects))}.\n"
        f"User command: {user_command!r}.\n"
        "Select the single available object that best fulfills the command. "
        "Return only JSON in the form {\"object\": \"exact name\"}. "
        "Return {\"object\": null} when none is appropriate."
    )
    messages = [{
        "role": "user",
        "content": [{"type": "text", "text": prompt}],
    }]
    inputs = qwen_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(qwen_model.device)

    # Decode only newly generated tokens, matching the image VLM helper.
    with torch.no_grad():
        generated_ids = qwen_model.generate(**inputs, max_new_tokens=64)
    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    response = qwen_processor.batch_decode(
        generated_ids, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()
    result = _parse_qwen_structure(response)
    selected = result.get("object") if isinstance(result, dict) else result
    if isinstance(selected, list):
        selected = selected[0] if selected else None

    return selected
    # Canonicalize spelling, then reject names outside the detected set.
    # canonical = {name.replace("_", " " ).lower(): name for name in available_objects}
    # selected_key = str(selected).replace("_", " " ).lower() if selected else ""
    # if selected_key not in canonical:
    #     raise RuntimeError(
    #         f"Qwen3-VL did not select an available object: {response!r}"
    #     )
    # return canonical[selected_key]


Available objects: ['Cup', 'Mustard_Bottle', 'Power_Drill', 'Soup_Can', 'Strawberry']


In [29]:
USER_COMMAND = "I want a can"
selected_object = select_detected_object(USER_COMMAND, detected_objects)
print(f"User command: {USER_COMMAND}")
print(f"Selected object: {selected_object}")

User command: I want a can
Selected object: Soup_Can


### 3. Load the selected cloud and generate a grasp

Sample only the selected object, visualize the best candidate with the saved scene, and write its command JSON.

In [4]:
from pathlib import Path

SCAN_DIR = Path("/home/hier-tony/Projects/vla_dual_arm/scan_output")
selected_object = "Soup_Can"

from grasping.grasp_sampling import (
    GripperGeometry, SamplingCriteria, build_gripper_commands,
    candidate_record, sample_grasps,
)
from scene_reconstruction.o3d_process import load_point_cloud_json, visualize_grasp

POINT_CLOUD_INDEX_PATH = SCAN_DIR / "pointclouds" / "index.json"
pointcloud_index = json.loads(POINT_CLOUD_INDEX_PATH.read_text(encoding="utf-8"))
object_record = pointcloud_index["objects"].get(selected_object)
if object_record is None:
    raise KeyError(f"{selected_object} has no saved point cloud")
selected_object_cloud = load_point_cloud_json(SCAN_DIR / object_record["path"])
saved_scene_cloud = load_point_cloud_json(SCAN_DIR / pointcloud_index["scene"])

# Configure and sample collision-filtered antipodal grasp candidates.
gripper_geometry = GripperGeometry(max_opening=0.1, outer_width=0.12, hand_depth=0.04, finger_reach=0.05, grasp_depth=0.02)
sampling_criteria = SamplingCriteria(
    samples=100, antipodal_angle_deg=35.0, voxel_size=0.005
)
selected_candidates = sample_grasps(
    selected_object_cloud,
    scene_cloud=saved_scene_cloud,
    geometry=gripper_geometry,
    criteria=sampling_criteria,
    preferred_approach=(0.0, 0.0, -1.0),
)
if not selected_candidates:
    raise RuntimeError(f"no valid grasp found for {selected_object}")

# Serialize the best command while retaining candidates for inspection.
best_candidate = selected_candidates[0]
selected_grasp_command = build_gripper_commands(
    best_candidate, object_name=selected_object
)
GRASP_OUTPUT_DIR = SCAN_DIR / "grasp_commands"
GRASP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
safe_name = selected_object.lower().replace(" " , "_").replace("/", "_")
grasp_output_path = GRASP_OUTPUT_DIR / f"{safe_name}.json"
grasp_output_path.write_text(
    json.dumps({
        "best_command": selected_grasp_command,
        "candidates": [
            candidate_record(candidate) for candidate in selected_candidates[:10]
        ],
    }, indent=2) + "\n",
    encoding="utf-8",
)
print(
    f"Best grasp: score={best_candidate.score:.3f}, "
    f"opening={best_candidate.required_opening * 1000:.1f} mm"
)
print(f"Saved command: {grasp_output_path}")

VISUALIZE_GRASP = True
if VISUALIZE_GRASP:
    visualize_grasp(
        saved_scene_cloud,
        best_candidate,
        object_cloud=selected_object_cloud,
        geometry=gripper_geometry,
        window_name=f"Best grasp: {selected_object}",
    )
selected_grasp_command


Best grasp: score=0.507, opening=73.2 mm
Saved command: /home/hier-tony/Projects/vla_dual_arm/scan_output/grasp_commands/soup_can.json


{'object': 'Soup_Can',
 'arm': 'left',
 'end_effector': 'left_ee',
 'planning_group': 'dual_arm',
 'score': 0.5073263146768312,
 'required_opening_m': 0.07315094491050853,
 'commands': [{'action': 'open_gripper', 'end_effector': 'left_ee'},
  {'action': 'move_cartesian',
   'planning_group': 'dual_arm',
   'pose': {'position_xyz': [0.3384457598911521,
     0.06304883823451753,
     0.19230656732745285],
    'quaternion_xyzw': [0.876074282659661,
     -0.4821499240409661,
     -0.0008620955258653261,
     -0.00495568370206629]}},
  {'action': 'move_cartesian',
   'planning_group': 'dual_arm',
   'pose': {'position_xyz': [0.33877258445137526,
     0.0640002795018551,
     0.09231162772938298],
    'quaternion_xyzw': [0.876074282659661,
     -0.4821499240409661,
     -0.0008620955258653261,
     -0.00495568370206629]}},
  {'action': 'close_gripper', 'end_effector': 'left_ee'}]}

### 4. Validate and optionally execute the gripper command

Run file
```bash
python3 scripts/grasping/grasp_test.py
```
Execution is disabled by default. Review the printed plan and set `EXECUTE_GRASP = True` only when the robot workspace is clear.